# Lab 5 — AUTO CDC

Practiced applying change events to a live target with `CREATE FLOW ... AUTO CDC INTO`:

* Loaded order **status changes** and **customer** change data (with `operation` + `timestamp` columns) as silver streaming tables
* AUTO CDC upserts each change into the target keyed on the entity id, sequenced by timestamp, applying `operation = 'DELETE'` as row deletes (SCD Type 1)
* Verified the results and debugged with the pipeline event log

In [0]:
-- Step 1: peek at the inferred schema of the raw order-status JSON before loading it.
-- DESCRIBE QUERY = a dry-run: shows columns/types without executing the query (LIMIT 0 reads no rows)
DESCRIBE QUERY
SELECT * FROM  read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/status/",
    format => "json"
) LIMIT 0;

In [0]:
-- Step 2: view ALL status-change events for order 75123 — later we can check that AUTO CDC replayed them in the right order
SELECT * FROM  read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/status/",
    format => "json"
)
where order_id = 75123

In [0]:
-- Orders silver streaming table (built by the pipeline with an EXPECT data-quality constraint)
SELECT * FROM databricks_labs.default.orders_silver_st;

In [0]:
-- Status silver streaming table: one row per status-change event for every order
SELECT * FROM databricks_labs.default.orders_status_silver_st

In [0]:
-- Enrich: LEFT JOIN orders with their status events → see each order's full lifecycle (placed → shipped → delivered …)
SELECT A.order_id,A.customer_id,A.order_timestamp,B.order_status,B.status_timestamp FROM databricks_labs.default.orders_silver_st as A
LEFT JOIN databricks_labs.default.orders_status_silver_st as B
ON A.order_id = B.order_id

In [0]:
-- Debugging: read the pipeline's event log for the status silver table — flow progress, errors, and data-quality metrics
SELECT
    timestamp,
    event_type,
    level,
    message
FROM event_log(
    TABLE(databricks_labs.default.orders_status_silver_st)
)
ORDER BY timestamp DESC
LIMIT 20;

In [0]:
-- Inspect the raw customer JSON: note the `operation` (insert/update/delete) and `timestamp` columns that drive AUTO CDC
select * from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/customer/",
    format => "json"
)

In [0]:
-- Verify the CDC target: customers_current holds ONLY the latest version of each customer (SCD Type 1 upserts + deletes)
SELECT COUNT(*) AS total_customers
FROM databricks_labs.default.customers_current;

In [0]:
-- Check the two customers that received UPDATE / DELETE change events — proves AUTO CDC applied them correctly
SELECT *
FROM databricks_labs.default.customers_current
WHERE customer_id IN (23225, 23617);